# Optional Colab development inference
Run the application and database in VS Code. This notebook runs only the inference gateway. Select an L4 (24GB) GPU runtime, or a larger GPU. Upload `server.py` and `requirements.txt` from the repository's `inference/` directory through the Files panel. Add `INFERENCE_API_KEY` in Colab Secrets and grant this notebook access. Model downloads and GPU availability depend on the runtime. This notebook defaults to Qwen/Qwen3.8-27B in 4-bit, non-thinking mode. GPU memory requirements depend on input length and runtime overhead; 4-bit loading still downloads the full checkpoint. The configuration cell rejects GPUs below 20 GiB usable capacity. Review the model license. No output cells or credentials are saved in this template.
Only use an approved HTTPS tunnel. The gateway also requires its bearer key. Colab is optional; offline and local Ollama modes work independently.

In [ ]:
from pathlib import Path
import shutil
folder = Path('/content/inference')
folder.mkdir(exist_ok=True)
for name in ('server.py', 'requirements.txt'):
    source = Path('/content') / name
    target = folder / name
    if source.exists():
        shutil.copyfile(source, target)
    if not target.exists():
        raise FileNotFoundError(f'Upload {name} using the Files panel first.')
print('Gateway files available.')

In [ ]:
%pip install --upgrade -r /content/inference/requirements.txt

In [ ]:
import os
from google.colab import userdata
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')
os.environ['INFERENCE_API_KEY'] = userdata.get('INFERENCE_API_KEY')
if not os.environ['INFERENCE_API_KEY']:
    raise RuntimeError('Set INFERENCE_API_KEY in Colab Secrets.')
os.environ['AI_MODEL'] = 'Qwen/Qwen3.8-27B'
# 27B NF4 requires a 24GB-class GPU such as L4.
gpu = torch.cuda.get_device_properties(0)
gpu_gib = gpu.total_memory / 1024**3
if gpu_gib < 20:
    raise RuntimeError(f'{gpu.name} has {gpu_gib:.1f} GiB VRAM. Select L4 (24GB) or a larger GPU for the 27B model.')
print(f'GPU: {gpu.name}; VRAM: {gpu_gib:.1f} GiB')
# Optional evaluated LoRA adapter path; leave empty for the original model.
os.environ['AI_LORA_ADAPTER'] = ''
os.environ['AI_LOAD_IN_4BIT'] = 'true'
os.environ['AI_ENABLE_THINKING'] = 'false'
os.environ['AI_TOKEN_TIMING'] = 'true'
os.environ['SC_ASSESS_OUTPUT_FORMAT'] = 'lines-v1'
os.environ['AI_ASSESS_MAX_NEW_TOKENS'] = '4096'
os.environ['AI_ASSESS_TOKENS_PER_QUESTION'] = '176'
os.environ['AI_ASSESS_MAX_INPUT_TOKENS'] = '6144'
os.environ['AI_QUESTIONS_MAX_NEW_TOKENS'] = '2048'
os.environ['AI_QUESTIONS_MAX_INPUT_TOKENS'] = '6144'
os.environ['AI_MAX_NEW_TOKENS'] = '8192'
print('Configuration ready; credentials are not printed.')

In [ ]:
import os
import subprocess
import sys
import time
import json
from urllib.request import Request, urlopen
from urllib.error import URLError
if 'gateway' in globals() and gateway.poll() is None:
    raise RuntimeError('Gateway is already running. Stop it before launching again.')
gateway_log = open('/content/gateway.log', 'a', buffering=1)
gateway = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'inference.server:app', '--host', '127.0.0.1', '--port', '9000', '--log-level', 'info'], cwd='/content', stdout=gateway_log, stderr=subprocess.STDOUT)
headers = {'Authorization': 'Bearer ' + os.environ['INFERENCE_API_KEY']}
for attempt in range(30):
    if gateway.poll() is not None:
        raise RuntimeError('Gateway exited. Check installed dependencies and configuration.')
    try:
        with urlopen(Request('http://127.0.0.1:9000/health', headers=headers), timeout=2) as response:
            status = json.load(response)
        break
    except URLError:
        time.sleep(1)
else:
    raise RuntimeError('Gateway did not become healthy.')
if 'departments' not in status.get('supported_operations', []):
    raise RuntimeError('This gateway is outdated. Upload the current inference/server.py, rerun the setup cell, and restart the gateway.')
if 'qwen3.8' not in status.get('qwen_hybrid_models', []):
    raise RuntimeError('Upload the updated server.py and restart the gateway; Qwen3.8 support is required.')
print('Gateway API', status.get('gateway_version'), '| department mapping supported')
print('Gateway started. Loading model weights may take several minutes.')
with urlopen(Request('http://127.0.0.1:9000/warmup', headers=headers, data=b'', method='POST'), timeout=1800) as response:
    ready = json.load(response)
print(ready['status'], ready['model'])
with urlopen(Request('http://127.0.0.1:9000/health', headers=headers), timeout=10) as response:
    health = json.load(response)
print({key: health.get(key) for key in ('gateway_version', 'model', 'thinking_enabled', 'load_in_4bit')})
print('Gateway logs: /content/gateway.log')


## Connect the local backend
Expose loopback port 9000 through your approved authenticated HTTPS tunnel. Tunnel tooling/accounts are intentionally not provisioned here. In the VS Code backend terminal set `AI_PROVIDER=http`, `AI_BASE_URL` to the tunnel HTTPS origin, `AI_API_KEY` to the same Colab secret, and `AI_MODEL` to `Qwen/Qwen3.8-27B`; then restart FastAPI. Never put credentials in a URL, repository, or notebook output.
From the project root, run `python scripts/check_inference.py` to check authentication/model readiness and a synthetic question request. The command reads environment variables and does not use company data. Run an assessment only after this check succeeds. For a full 93-control generation, allow several minutes; afterward run the measurement cell below to see the latest prompt/output tokens and generation seconds.
Colab sessions and tunnel addresses can expire. Stop the tunnel when finished. To return to local operation, set `AI_PROVIDER=ollama` or `offline` and restart FastAPI.

## Measure latest inference
Run this after the local app finishes generating the questionnaire. It shows measurements for the latest model request attempt.

In [ ]:
import json
from urllib.request import Request, urlopen
headers = {'Authorization': 'Bearer ' + os.environ['INFERENCE_API_KEY']}
with urlopen(Request('http://127.0.0.1:9000/health', headers=headers), timeout=10) as response:
    health = json.load(response)
print(json.dumps(health.get('last_inference'), indent=2))


In [ ]:
import os
import subprocess
if 'gateway' in globals() and gateway.poll() is None:
    gateway.terminate()
    try:
        gateway.wait(timeout=15)
    except subprocess.TimeoutExpired:
        gateway.kill()
        gateway.wait(timeout=5)
if 'gateway_log' in globals() and not gateway_log.closed:
    gateway_log.close()
os.environ.pop('INFERENCE_API_KEY', None)
headers = {}
print('Gateway stopped. Stop your HTTPS tunnel separately.')
